# EfficientNet Comparison

## Overview
This notebook trains an EfficientNet-B0 model as an alternative architecture 
to compare against the ResNet50 baseline, following the professor's 
recommendation to test different model architectures.

## Why EfficientNet?
EfficientNet-B0 is significantly smaller than ResNet50 (20.5MB vs 97.8MB, 
~5x fewer parameters) but often achieves comparable performance, making it 
an interesting efficiency comparison.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from google.colab import drive

In [2]:
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_dir = "/content/drive/MyDrive/brain-tumor-data/Training"
test_dir = "/content/drive/MyDrive/brain-tumor-data/Testing"

classes = ["glioma", "meningioma", "notumor", "pituitary"]
num_classes = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

print("Setup done!")

Mounted at /content/drive
Using device: cuda
Setup done!


In [3]:
class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.images = []
        self.labels = []
        
        for cls in classes:
            cls_path = os.path.join(root_dir, cls)
            for img_file in os.listdir(cls_path):
                self.images.append(os.path.join(cls_path, img_file))
                self.labels.append(class_to_idx[cls])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]
    

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = BrainTumorDataset(train_dir, transform=train_transform)
test_dataset = BrainTumorDataset(test_dir, transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

Training samples: 5600
Testing samples: 1600


In [5]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer for our 4 classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device)
print(model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 113MB/s] 


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)


## Training
Same transfer learning approach as the ResNet50 baseline: pretrained 
ImageNet weights, all layers frozen except the final classifier.

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier[1].parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(train_loader):.4f} Accuracy: {acc:.2f}%")

torch.save(model.state_dict(), "/content/drive/MyDrive/brain-tumor-data/efficientnet_model.pth")
print("EfficientNet saved!")

Epoch [1/10] Loss: 0.6636 Accuracy: 77.75%
Epoch [2/10] Loss: 0.4548 Accuracy: 83.98%
Epoch [3/10] Loss: 0.4125 Accuracy: 85.52%
Epoch [4/10] Loss: 0.3950 Accuracy: 85.45%
Epoch [5/10] Loss: 0.3826 Accuracy: 86.41%
Epoch [6/10] Loss: 0.3687 Accuracy: 86.62%
Epoch [7/10] Loss: 0.3588 Accuracy: 87.05%
Epoch [8/10] Loss: 0.3538 Accuracy: 86.96%
Epoch [9/10] Loss: 0.3432 Accuracy: 87.71%
Epoch [10/10] Loss: 0.3574 Accuracy: 86.64%
EfficientNet saved!


## Evaluation

**Results:** 84.50% test accuracy, slightly lower than ResNet50 baseline 
(85.75%) despite being 5x smaller. Glioma recall (68%) remained the 
weakest class, similar to the ResNet50 baseline pattern.

In [11]:
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")
print(classification_report(all_labels, all_preds, target_names=classes))

Test Accuracy: 84.50%
              precision    recall  f1-score   support

      glioma       0.94      0.68      0.79       400
  meningioma       0.76      0.78      0.77       400
     notumor       0.79      0.99      0.88       400
   pituitary       0.93      0.93      0.93       400

    accuracy                           0.84      1600
   macro avg       0.86      0.84      0.84      1600
weighted avg       0.86      0.84      0.84      1600



## Binary EfficientNet for Stage 1

To compare with the ResNet50-based Stage 1 model, a binary version of 
EfficientNet was also trained for tumor vs no tumor detection.

**Result:** 94.31% test accuracy. While EfficientNet achieved higher 
notumor recall (99% vs ~97%), it had lower tumor recall (93% vs ~96%), 
making ResNet50's Stage 1 the safer choice clinically since it misses 
fewer actual tumors.

In [5]:

binary_label = {"glioma": 1, "meningioma": 1, "notumor": 0, "pituitary": 1}
all_classes = ["glioma", "meningioma", "notumor", "pituitary"]

class BinaryDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.images = []
        self.labels = []
        
        for cls in all_classes:
            cls_path = os.path.join(root_dir, cls)
            for img_file in os.listdir(cls_path):
                self.images.append(os.path.join(cls_path, img_file))
                self.labels.append(binary_label[cls])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

binary_train = BinaryDataset(train_dir, transform=train_transform)
binary_test = BinaryDataset(test_dir, transform=test_transform)
binary_train_loader = DataLoader(binary_train, batch_size=BATCH_SIZE, shuffle=True)
binary_test_loader = DataLoader(binary_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"Binary training samples: {len(binary_train)}")
print(f"Binary testing samples: {len(binary_test)}")

Binary training samples: 5600
Binary testing samples: 1600


In [7]:
binary_efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Freeze all layers
for param in binary_efficientnet.parameters():
    param.requires_grad = False

# Replace final layer with binary output
binary_efficientnet.classifier[1] = nn.Linear(binary_efficientnet.classifier[1].in_features, 2)
binary_efficientnet = binary_efficientnet.to(device)

# Train
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(binary_efficientnet.classifier[1].parameters(), lr=LEARNING_RATE)

print("=== Training Binary EfficientNet (Tumor vs No Tumor) ===")
for epoch in range(EPOCHS):
    binary_efficientnet.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in binary_train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = binary_efficientnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(binary_train_loader):.4f} Accuracy: {acc:.2f}%")

torch.save(binary_efficientnet.state_dict(), "/content/drive/MyDrive/brain-tumor-data/binary_efficientnet.pth")
print("Binary EfficientNet saved!")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 166MB/s]


=== Training Binary EfficientNet (Tumor vs No Tumor) ===
Epoch [1/10] Loss: 0.2037 Accuracy: 92.14%
Epoch [2/10] Loss: 0.1308 Accuracy: 95.54%
Epoch [3/10] Loss: 0.1108 Accuracy: 96.21%
Epoch [4/10] Loss: 0.1111 Accuracy: 95.89%
Epoch [5/10] Loss: 0.0993 Accuracy: 96.54%
Epoch [6/10] Loss: 0.0962 Accuracy: 96.50%
Epoch [7/10] Loss: 0.0959 Accuracy: 96.45%
Epoch [8/10] Loss: 0.0867 Accuracy: 96.86%
Epoch [9/10] Loss: 0.0874 Accuracy: 96.98%
Epoch [10/10] Loss: 0.0860 Accuracy: 97.18%
Binary EfficientNet saved!


In [6]:
#For when model is already trained
binary_efficientnet = models.efficientnet_b0(weights=None)
binary_efficientnet.classifier[1] = nn.Linear(binary_efficientnet.classifier[1].in_features, 2)
binary_efficientnet.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/binary_efficientnet.pth", map_location=device))
binary_efficientnet = binary_efficientnet.to(device)
binary_efficientnet.eval()
print("Binary EfficientNet loaded!")

Binary EfficientNet loaded!


In [7]:
binary_efficientnet.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in binary_test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = binary_efficientnet(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f"Binary EfficientNet Test Accuracy: {accuracy:.2f}%")
print(classification_report(all_labels, all_preds, target_names=["notumor", "tumor"]))

Binary EfficientNet Test Accuracy: 94.31%
              precision    recall  f1-score   support

     notumor       0.82      0.99      0.90       400
       tumor       1.00      0.93      0.96      1200

    accuracy                           0.94      1600
   macro avg       0.91      0.96      0.93      1600
weighted avg       0.95      0.94      0.94      1600



## Fine-tuning EfficientNet

Following the same fine-tuning strategy used for ResNet50, the last block 
of EfficientNet's feature extractor was unfrozen and retrained.

**Result:** Test accuracy improved from 84.50% to 88.75%. However, glioma 
recall remained unchanged at 69%, suggesting EfficientNet's architecture 
may be less suited to capturing the subtle features that distinguish 
glioma from other tumor types.

In [6]:
# Load saved EfficientNet
finetuned_efficientnet = models.efficientnet_b0(weights=None)
finetuned_efficientnet.classifier[1] = nn.Linear(finetuned_efficientnet.classifier[1].in_features, num_classes)
finetuned_efficientnet.load_state_dict(torch.load("/content/drive/MyDrive/brain-tumor-data/efficientnet_model.pth", map_location=device))
finetuned_efficientnet = finetuned_efficientnet.to(device)

# Freeze all layers first
for param in finetuned_efficientnet.parameters():
    param.requires_grad = False

# Unfreeze last block and classifier
for param in finetuned_efficientnet.features[-1].parameters():
    param.requires_grad = True
for param in finetuned_efficientnet.classifier.parameters():
    param.requires_grad = True

# Check trainable parameters
trainable = sum(p.numel() for p in finetuned_efficientnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in finetuned_efficientnet.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

# Train
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, finetuned_efficientnet.parameters()),
    lr=0.0001
)

print("=== Fine-tuning EfficientNet ===")
for epoch in range(EPOCHS):
    finetuned_efficientnet.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = finetuned_efficientnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(train_loader):.4f} Accuracy: {acc:.2f}%")

torch.save(finetuned_efficientnet.state_dict(), "/content/drive/MyDrive/brain-tumor-data/efficientnet_finetuned.pth")
print("Fine-tuned EfficientNet saved!")

Trainable parameters: 417,284 / 4,012,672
=== Fine-tuning EfficientNet ===
Epoch [1/10] Loss: 0.3240 Accuracy: 88.02%
Epoch [2/10] Loss: 0.3234 Accuracy: 88.02%
Epoch [3/10] Loss: 0.2948 Accuracy: 89.32%
Epoch [4/10] Loss: 0.2938 Accuracy: 89.39%
Epoch [5/10] Loss: 0.2751 Accuracy: 89.95%
Epoch [6/10] Loss: 0.2711 Accuracy: 90.11%
Epoch [7/10] Loss: 0.2637 Accuracy: 90.61%
Epoch [8/10] Loss: 0.2486 Accuracy: 90.55%
Epoch [9/10] Loss: 0.2438 Accuracy: 90.93%
Epoch [10/10] Loss: 0.2410 Accuracy: 90.66%
Fine-tuned EfficientNet saved!


In [7]:
finetuned_efficientnet.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = finetuned_efficientnet(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f"Fine-tuned EfficientNet Test Accuracy: {accuracy:.2f}%")
print(classification_report(all_labels, all_preds, target_names=classes))

Fine-tuned EfficientNet Test Accuracy: 88.75%
              precision    recall  f1-score   support

      glioma       0.98      0.69      0.81       400
  meningioma       0.80      0.89      0.84       400
     notumor       0.88      0.99      0.93       400
   pituitary       0.93      0.97      0.95       400

    accuracy                           0.89      1600
   macro avg       0.90      0.89      0.88      1600
weighted avg       0.90      0.89      0.88      1600

